In [1]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from collections import OrderedDict
import os, gc, random, math, warnings
warnings.filterwarnings('ignore')

for _f in ['/kaggle/working/checkpoint.pth', '/kaggle/working/best_model.pth']:
    if os.path.exists(_f): os.remove(_f)

def set_seeds(seed=123):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    random.seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False
    os.environ['PYTHONHASHSEED'] = str(seed)

set_seeds(123)
print("✅ Imports done")
print("CUDA:", torch.cuda.is_available())
print("GPU :", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")


✅ Imports done
CUDA: True
GPU : Tesla T4


In [2]:
class Config:
    # Try all possible Phase 2 dataset paths
    _PATHS = [
        '/kaggle/input/competitions/anrf-aise-hack-phase-2-theme-2-pollution-forecasting-iitd',
        '/kaggle/input/anrf-aise-hack-phase-2-theme-2-pollution-forecasting-iitd',
        '/kaggle/input/aisehack-theme-2',
    ]
    _BASE = next((p for p in _PATHS if os.path.exists(p)), _PATHS[0])
    # Handle both direct and nested dataset structures
    DATA_ROOT   = (_BASE + '/aisehack-theme-2/raw'
                   if os.path.exists(_BASE + '/aisehack-theme-2/raw')
                   else _BASE + '/raw')
    TEST_PATH   = (_BASE + '/aisehack-theme-2/test_in'
                   if os.path.exists(_BASE + '/aisehack-theme-2/test_in')
                   else _BASE + '/test_in')
    OUTPUT_PATH = '/kaggle/working/preds.npy'
    MODEL_PATH  = '/kaggle/working/best_model.pth'
    CKPT_PATH   = '/kaggle/working/checkpoint.pth'

    _all_months = os.listdir(DATA_ROOT) if os.path.exists(DATA_ROOT) else []
    MONTHS = sorted([m for m in _all_months if not m.endswith('.npy')])

    # ── 15 features (proven best in Phase 1) ─────────────────
    ALL_AUX = ['q2', 't2', 'u10', 'v10', 'swdown', 'pblh', 'psfc', 'rain',
               'NOx', 'SO2', 'NH3', 'NMVOC_e', 'NMVOC_finn', 'bio', 'PM25']

    # ── PHASE 2 KEY CHANGE ────────────────────────────────────
    # Phase 1: AUX_WIN=26 (10 history + 16 future met)
    # Phase 2: AUX_WIN=10 (10 history ONLY — no future met available)
    PM25_IN  = 10
    AUX_WIN  = 10   # CHANGED from 26 — only historical window available
    HORIZON  = 16
    STRIDE   = 1

    # ── Model (proven Phase 1 architecture) ───────────────────
    FNO_MODES = 7
    FNO_WIDTH = 8
    NUM_FNO   = 1
    HIDDEN    = 60
    DROPOUT   = 0.1

    # Hotspot: IGP upweighted
    HOT_R1, HOT_R2 = 68, 94
    HOT_C1, HOT_C2 = 39, 80
    HOT_WEIGHT      = 2.0

    # ── Training ──────────────────────────────────────────────
    MAX_EPOCHS   = 100
    ES_PATIENCE  = 20
    BATCH_SIZE   = 4
    LR           = 2e-4
    WEIGHT_DECAY = 2e-5
    GRAD_CLIP    = 1.0

    # ── Phase 2 Loss weights ──────────────────────────────────
    # SMAPE aligns directly with evaluation metric
    # Episode upweighting: episodic cells get higher loss weight
    SMAPE_WT      = 1.0    # global SMAPE loss
    EPISODE_WT    = 2.0    # extra weight on episodic grid points
    PHYSICS_WT    = 0.03   # physics advection constraint
    EPISODE_SIGMA = 1.5    # threshold: PM25 > mean + sigma*std = episode

    VAL_FRAC      = 0.15

    SEED   = 123
    DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

cfg = Config()
NAUX    = len(cfg.ALL_AUX)   # 15
# LIFT_IN: PM25_IN(10) + AUX_WIN(10)*NAUX(15) = 160
LIFT_IN = cfg.PM25_IN + cfg.AUX_WIN * NAUX   # 160

print(f"✅ Config ready  [Phase 2 — Episode-Aware Forecasting]")
print(f"Device      : {cfg.DEVICE}")
print(f"AUX_WIN     : {cfg.AUX_WIN}  (was 26 in Phase 1 — now historical only)")
print(f"LIFT_IN     : {LIFT_IN}  (was 400 in Phase 1)")
print(f"NAUX        : {NAUX}  (15 features — all met + emissions)")
print(f"Output shape: (218, 140, 124, 16)")
print(f"Metric      : SMAPE + EpisodeCorr (higher = better)")


✅ Config ready  [Phase 2 — Episode-Aware Forecasting]
Device      : cuda
AUX_WIN     : 10  (was 26 in Phase 1 — now historical only)
LIFT_IN     : 160  (was 400 in Phase 1)
NAUX        : 15  (15 features — all met + emissions)
Output shape: (218, 140, 124, 16)
Metric      : SMAPE + EpisodeCorr (higher = better)


In [3]:
print("Computing normalisation stats...")

all_pm25_log = []
for month in cfg.MONTHS:
    arr = np.load(os.path.join(cfg.DATA_ROOT, month, 'cpm25.npy')).astype(np.float32)
    all_pm25_log.append(np.log1p(arr))
all_pm25_log = np.concatenate(all_pm25_log)
PM25_LOG_MIN = float(all_pm25_log.min())
PM25_LOG_MAX = float(all_pm25_log.max())
PM25_LOG_RNG = PM25_LOG_MAX - PM25_LOG_MIN
del all_pm25_log; gc.collect()
print(f"  PM2.5 log: {PM25_LOG_MIN:.4f} → {PM25_LOG_MAX:.4f}")

# Aux: raw min-max (same as Phase 1 best model)
FEAT_MINS, FEAT_MAXS = {}, {}
for feat in cfg.ALL_AUX:
    vals = []
    for month in cfg.MONTHS:
        p = os.path.join(cfg.DATA_ROOT, month, f'{feat}.npy')
        if os.path.exists(p):
            vals.append(np.load(p).astype(np.float32))
    if vals:
        v = np.concatenate(vals)
        FEAT_MINS[feat] = float(v.min())
        FEAT_MAXS[feat] = float(v.max())
        del v
    else:
        FEAT_MINS[feat] = 0.0; FEAT_MAXS[feat] = 1.0
    print(f"  {feat}: {FEAT_MINS[feat]:.4f} → {FEAT_MAXS[feat]:.4f}")

gc.collect()
print("✅ Stats done")


Computing normalisation stats...
  PM2.5 log: 0.0012 → 7.9553
  q2: 0.0000 → 0.0379
  t2: 226.4993 → 322.6777
  u10: -25.0179 → 29.0259
  v10: -25.6960 → 31.0424
  swdown: 0.0000 → 1287.9806
  pblh: 53.6355 → 6064.9565
  psfc: 48015.4492 → 101961.3125
  rain: 0.0000 → 397.0356
  NOx: 0.0000 → 0.0000
  SO2: 0.0000 → 0.0000
  NH3: 0.0000 → 0.0000
  NMVOC_e: 0.0000 → 0.0000
  NMVOC_finn: 0.0000 → 0.0000
  bio: 0.0000 → 0.0000
  PM25: 0.0000 → 0.0000
✅ Stats done


In [4]:
def norm_pm25(arr):
    return ((np.log1p(arr) - PM25_LOG_MIN) / (PM25_LOG_RNG + 1e-8)).astype(np.float32)

def denorm_pm25_np(arr):
    return np.expm1(np.clip(arr * (PM25_LOG_RNG + 1e-8) + PM25_LOG_MIN, 0, None))

def load_month(month):
    base    = os.path.join(cfg.DATA_ROOT, month)
    arr_raw = np.load(os.path.join(base, 'cpm25.npy')).astype(np.float32)
    T       = arr_raw.shape[0]
    data    = {'cpm25': norm_pm25(arr_raw), 'cpm25_raw': arr_raw}
    for feat in cfg.ALL_AUX:
        p   = os.path.join(base, f'{feat}.npy')
        arr = np.load(p).astype(np.float32) if os.path.exists(p) \
              else np.zeros((T, 140, 124), dtype=np.float32)
        data[feat] = ((arr - FEAT_MINS[feat]) /
                      (FEAT_MAXS[feat] - FEAT_MINS[feat] + 1e-8)).astype(np.float32)
    return data

def get_t0_list_temporal(T):
    # Phase 2: need PM25_IN + HORIZON timesteps (only 10+16=26, not 26+16)
    total   = cfg.PM25_IN + cfg.HORIZON
    all_idx = list(range(0, T - total + 1, cfg.STRIDE))
    split   = int(len(all_idx) * (1 - cfg.VAL_FRAC))
    return all_idx[:split], all_idx[split:]

def compute_episode_mask(pm25_raw_seq):
    """Flag episodic grid points: cells where PM25 > local mean + sigma*std.
    Returns boolean mask shape (H, W) — True = episodic.
    """
    mean = pm25_raw_seq.mean(axis=0)   # (H, W)
    std  = pm25_raw_seq.std(axis=0)    # (H, W)
    # Flag cells that spike significantly above local baseline
    # Use the TARGET window (future 16h) for episode detection
    return pm25_raw_seq > (mean + cfg.EPISODE_SIGMA * std + 1e-6)  # (T, H, W)

class PM25Dataset(Dataset):
    def __init__(self, month_arrays, indices):
        self.data    = month_arrays
        self.indices = indices
        self.u_idx   = cfg.ALL_AUX.index('u10')
        self.v_idx   = cfg.ALL_AUX.index('v10')

    def __len__(self): return len(self.indices)

    def __getitem__(self, i):
        m, t0 = self.indices[i]
        pm25, aux_stack, pm25_raw = self.data[m]

        # ── Phase 2: AUX_WIN=10, so aux history = same window as PM25 ──
        x_pm  = pm25[t0 : t0 + cfg.PM25_IN]              # (10, H, W)
        x_aux = aux_stack[t0 : t0 + cfg.AUX_WIN]         # (10, 15, H, W)
        y     = pm25[t0 + cfg.PM25_IN :
                     t0 + cfg.PM25_IN + cfg.HORIZON]      # (16, H, W)

        # ── Episode mask for target window ────────────────────
        raw_target = pm25_raw[t0 + cfg.PM25_IN :
                               t0 + cfg.PM25_IN + cfg.HORIZON]  # (16, H, W)
        raw_hist   = pm25_raw[t0 : t0 + cfg.PM25_IN]           # (10, H, W)
        # Episodic if target PM25 > mean_hist + sigma*std_hist
        mean_h = raw_hist.mean(axis=0)
        std_h  = raw_hist.std(axis=0) + 1e-6
        ep_mask = (raw_target > (mean_h + cfg.EPISODE_SIGMA * std_h)).astype(np.float32)  # (16, H, W)

        # Wind for physics loss (use last known aux step)
        fut_u = aux_stack[t0 + cfg.AUX_WIN - 1, self.u_idx]   # (H, W) last known
        fut_v = aux_stack[t0 + cfg.AUX_WIN - 1, self.v_idx]

        return (torch.from_numpy(np.ascontiguousarray(x_pm)),
                torch.from_numpy(np.ascontiguousarray(x_aux)),
                torch.from_numpy(np.ascontiguousarray(y)),
                torch.from_numpy(np.ascontiguousarray(fut_u)),
                torch.from_numpy(np.ascontiguousarray(fut_v)),
                torch.from_numpy(np.ascontiguousarray(ep_mask)))

print("✅ Data pipeline ready")
print(f"  AUX_WIN={cfg.AUX_WIN} — using historical met only (no future forcing)")
print(f"  Episode detection: PM25 > mean_hist + {cfg.EPISODE_SIGMA}×std_hist")


✅ Data pipeline ready
  AUX_WIN=10 — using historical met only (no future forcing)
  Episode detection: PM25 > mean_hist + 1.5×std_hist


In [5]:
# ── Proven Phase 1 architecture with Phase 2 modifications ──
# Key changes:
# 1. LIFT_IN = 160 (was 400) — 10 hist PM25 + 10 hist aux × 15 feats
# 2. AR decoder uses last-known met (no future forcing available)
# 3. Direct head unchanged

class SpectralConv2d(nn.Module):
    def __init__(self, in_ch, out_ch, modes1, modes2):
        super().__init__()
        self.modes1 = modes1; self.modes2 = modes2
        scale = 1 / (in_ch * out_ch)
        self.weights1 = nn.Parameter(scale * torch.rand(in_ch, out_ch, modes1, modes2, dtype=torch.cfloat))
        self.weights2 = nn.Parameter(scale * torch.rand(in_ch, out_ch, modes1, modes2, dtype=torch.cfloat))

    def compl_mul2d(self, x, w):
        return torch.einsum('bixy,ioxy->boxy', x, w)

    def forward(self, x):
        B, C, H, W = x.shape
        x_ft   = torch.fft.rfft2(x.float())
        out_ft = torch.zeros(B, self.weights1.shape[1], H, W//2+1, device=x.device, dtype=torch.cfloat)
        out_ft[:, :, :self.modes1, :self.modes2] = self.compl_mul2d(x_ft[:, :, :self.modes1, :self.modes2], self.weights1)
        out_ft[:, :, -self.modes1:, :self.modes2] = self.compl_mul2d(x_ft[:, :, -self.modes1:, :self.modes2], self.weights2)
        return torch.fft.irfft2(out_ft, s=(H, W))

class FNOBlock(nn.Module):
    def __init__(self, width, modes):
        super().__init__()
        self.spec = SpectralConv2d(width, width, modes, modes)
        self.conv = nn.Conv2d(width, width, 1)
        self.norm = nn.GroupNorm(8, width)
    def forward(self, x):
        return F.gelu(self.norm(self.spec(x) + self.conv(x)))

class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1), nn.GroupNorm(8, out_ch), nn.GELU(),
            nn.Dropout2d(cfg.DROPOUT),
            nn.Conv2d(out_ch, out_ch, 3, padding=1), nn.GroupNorm(8, out_ch), nn.GELU(),
        )
    def forward(self, x): return self.block(x)

class PTM_FNO_Phase2(nn.Module):
    def __init__(self):
        super().__init__()
        W = cfg.FNO_WIDTH  # 8

        # Lifting: PM25_IN(10) + AUX_WIN(10)*NAUX(15) = 160
        self.lift = nn.Sequential(
            nn.Conv2d(LIFT_IN, W*2, 1), nn.GELU(),
            nn.Conv2d(W*2, W, 1))

        self.fno_blocks = nn.Sequential(*[FNOBlock(W, cfg.FNO_MODES) for _ in range(cfg.NUM_FNO)])

        self.enc1 = DoubleConv(W,       64)
        self.enc2 = DoubleConv(64,     128)
        self.bot  = DoubleConv(128,    256)
        self.dec2 = DoubleConv(256+128,128)
        self.dec1 = DoubleConv(128+64,  64)
        self.fuse = nn.Conv2d(W+64, W, 1)
        self.pool = nn.MaxPool2d(2, ceil_mode=True)
        self.up   = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False)

        # ── PHASE 2 AR DECODER ────────────────────────────────
        # No future met available — use state + last_known_met + cur_pm25
        # Input: W(8) + NAUX(15) + 1 = 24
        self.step_conv = nn.Sequential(
            nn.Conv2d(W + NAUX + 1, cfg.HIDDEN, 3, padding=1), nn.GELU(),
            nn.Conv2d(cfg.HIDDEN, cfg.HIDDEN,   3, padding=1), nn.GELU(),
            nn.Conv2d(cfg.HIDDEN, 1, 1))

        # Direct head: state → all 16 steps at once
        self.direct_head = nn.Sequential(
            nn.Conv2d(W, cfg.HIDDEN, 3, padding=1), nn.GELU(),
            nn.Conv2d(cfg.HIDDEN, cfg.HORIZON, 1))

    def encode(self, pm25_hist, aux):
        B = pm25_hist.shape[0]
        aux_flat = aux.view(B, cfg.AUX_WIN * NAUX, 140, 124)  # (B, 150, H, W)
        x = self.lift(torch.cat([pm25_hist, aux_flat], 1))    # (B, W, H, W)
        fno_out = self.fno_blocks(x)
        e1 = self.enc1(x); e2 = self.enc2(self.pool(e1))
        b  = self.bot(self.pool(e2))
        d2 = self.dec2(torch.cat([self.up(b), e2], 1))
        d1 = self.dec1(torch.cat([self.up(d2), e1], 1))
        return F.gelu(self.fuse(torch.cat([fno_out, d1], 1)))

    def forward(self, pm25_hist, aux):
        state = self.encode(pm25_hist, aux)
        cur   = pm25_hist[:, -1:, :, :]

        # ── Last known met as proxy for future forcing ─────────
        # aux[:, -1, :, :] = most recent available met (t=9)
        # This is the best available estimate of near-future conditions
        last_met = aux[:, -1, :, :]   # (B, NAUX, H, W)

        preds = []
        for t in range(cfg.HORIZON):
            # Use last known met instead of future met
            delta = self.step_conv(torch.cat([state, last_met, cur], 1))
            cur   = cur + delta
            preds.append(cur)

        ar_out     = torch.cat(preds, 1)     # (B, 16, H, W)
        direct_out = self.direct_head(state) # (B, 16, H, W)
        return 0.6 * ar_out + 0.4 * direct_out


# Smoke test
_m = PTM_FNO_Phase2().to(cfg.DEVICE)
_p = torch.randn(1, cfg.PM25_IN, 140, 124).to(cfg.DEVICE)
_a = torch.randn(1, cfg.AUX_WIN, NAUX, 140, 124).to(cfg.DEVICE)
_o = _m(_p, _a)
assert _o.shape == (1, cfg.HORIZON, 140, 124), f"Shape: {_o.shape}"
print(f"✅ PTM_FNO_Phase2 ready | Params: {sum(p.numel() for p in _m.parameters()):,}")
print(f"   LIFT_IN={LIFT_IN}  (PM25_IN=10 + AUX_WIN=10 × NAUX=15)")
print(f"   AR decoder: state + last_known_met + cur_pm25 → delta")
del _m, _p, _a, _o


✅ PTM_FNO_Phase2 ready | Params: 1,949,065
   LIFT_IN=160  (PM25_IN=10 + AUX_WIN=10 × NAUX=15)
   AR decoder: state + last_known_met + cur_pm25 → delta


In [6]:
# ── Hotspot mask ─────────────────────────────────────────────
_hot_mask = torch.ones(1, 1, 140, 124)
_hot_mask[:, :, cfg.HOT_R1:cfg.HOT_R2, cfg.HOT_C1:cfg.HOT_C2] = cfg.HOT_WEIGHT
HOT_MASK = _hot_mask.to(cfg.DEVICE)

def smape_loss(pred, target, eps=1.0):
    """SMAPE — directly matches evaluation metric.
    eps prevents division by zero at near-zero concentrations.
    """
    denom = 0.5 * (pred.abs() + target.abs()) + eps
    return (pred - target).abs() / denom

def episode_weighted_loss(pred, target, ep_mask):
    """SMAPE with episode upweighting + hotspot mask.
    ep_mask: (B, 16, H, W) float, 1=episodic cell, 0=normal
    """
    base_smape = smape_loss(pred, target)   # (B, 16, H, W)
    # Upweight episodic cells
    episode_weight = 1.0 + cfg.EPISODE_WT * ep_mask
    # Upweight IGP hotspot
    spatial_weight = HOT_MASK   # (1, 1, H, W)
    return (base_smape * episode_weight * spatial_weight).mean()

def physics_loss(pred, wind_u, wind_v):
    """Penalise PM2.5 gradients that oppose wind direction.
    pred:   (B, 16, H, W)
    wind_u: (B, H, W) — last known zonal wind
    wind_v: (B, H, W) — last known meridional wind
    """
    B, T, H, W = pred.shape
    dpm_dx = pred[:, :, :, 1:] - pred[:, :, :, :-1]   # (B, T, H, W-1)
    dpm_dy = pred[:, :, 1:, :] - pred[:, :, :-1, :]   # (B, T, H-1, W)
    # Expand wind from (B,H,W) → (B,T,H,W), then slice to gradient sizes
    u = wind_u.unsqueeze(1).expand(B, T, H, W)         # (B,T,H,W) — 4D ✅
    v = wind_v.unsqueeze(1).expand(B, T, H, W)         # (B,T,H,W) — 4D ✅
    return ((dpm_dx * u[:, :, :, :dpm_dx.shape[3]].sign()).clamp(max=0).abs().mean() +
            (dpm_dy * v[:, :, :dpm_dy.shape[2], :].sign()).clamp(max=0).abs().mean())

def total_loss(pred, target, wu, wv, ep_mask):
    return (cfg.SMAPE_WT   * episode_weighted_loss(pred, target, ep_mask) +
            cfg.PHYSICS_WT * physics_loss(pred, wu, wv))

def val_metric_smape(pred_norm, target_norm):
    """Validation metric aligned with Phase 2 evaluation."""
    p = denorm_pm25_np(pred_norm.detach().cpu().float().numpy())
    t = denorm_pm25_np(target_norm.detach().cpu().float().numpy())
    smape = np.mean(np.abs(p - t) / (0.5 * (np.abs(p) + np.abs(t)) + 1.0))
    return float(smape)   # lower is better

print("✅ Phase 2 loss ready")
print(f"   SMAPE loss weight      : {cfg.SMAPE_WT}")
print(f"   Episode upweight       : {cfg.EPISODE_WT}× on episodic cells")
print(f"   IGP hotspot upweight   : {cfg.HOT_WEIGHT}×")
print(f"   Physics loss weight    : {cfg.PHYSICS_WT}")


✅ Phase 2 loss ready
   SMAPE loss weight      : 1.0
   Episode upweight       : 2.0× on episodic cells
   IGP hotspot upweight   : 2.0×
   Physics loss weight    : 0.03


In [7]:
print("Loading training data...")
month_arrays = []; train_indices = []; val_indices = []

for month in cfg.MONTHS:
    mpath = os.path.join(cfg.DATA_ROOT, month)
    if not os.path.exists(mpath): continue
    d       = load_month(month)
    pm25    = d['cpm25']
    pm25_raw= d['cpm25_raw']
    aux     = np.stack([d[f] for f in cfg.ALL_AUX], axis=1)  # (T, 15, H, W)
    midx    = len(month_arrays)
    month_arrays.append((pm25, aux, pm25_raw))
    tr, va  = get_t0_list_temporal(pm25.shape[0])
    train_indices.extend([(midx, t) for t in tr])
    val_indices.extend(  [(midx, t) for t in va])
    print(f"  {month}: train={len(tr)} val={len(va)}")

train_ds     = PM25Dataset(month_arrays, train_indices)
val_ds       = PM25Dataset(month_arrays, val_indices)
train_loader = DataLoader(train_ds, cfg.BATCH_SIZE, shuffle=True,
                          num_workers=2, pin_memory=True, drop_last=True)
val_loader   = DataLoader(val_ds, cfg.BATCH_SIZE*2, shuffle=False,
                          num_workers=2, pin_memory=True)
print(f"\n✅ Train: {len(train_ds)} | Val: {len(val_ds)}")


Loading training data...
  APRIL_16: train=586 val=104
  DEC_16: train=606 val=108
  JULY_16: train=606 val=108
  OCT_16: train=606 val=108

✅ Train: 2404 | Val: 428


In [8]:
from tqdm.notebook import tqdm

model     = PTM_FNO_Phase2().to(cfg.DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.LR, weight_decay=cfg.WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=cfg.MAX_EPOCHS, eta_min=1e-6)

best_val   = float('inf')
best_state = None
start_epoch = 1
es_counter  = 0

if os.path.exists(cfg.CKPT_PATH):
    try:
        ck = torch.load(cfg.CKPT_PATH, map_location=cfg.DEVICE)
        model.load_state_dict(ck['model']); optimizer.load_state_dict(ck['optimizer'])
        scheduler.load_state_dict(ck['scheduler'])
        best_val=ck['best_val']; best_state=ck['best_state']
        start_epoch=ck['epoch']+1; es_counter=ck.get('es_counter',0)
        print(f"✅ Resumed epoch {ck['epoch']} | best={best_val:.4f}")
    except Exception as e:
        print(f"⚠️ Bad checkpoint ({e}), fresh start"); os.remove(cfg.CKPT_PATH)
else:
    print(f"Fresh start | seed=123 | {cfg.MAX_EPOCHS} epochs | ES={cfg.ES_PATIENCE}")

ebar = tqdm(range(start_epoch, cfg.MAX_EPOCHS+1), desc='Training', unit='ep',
            bar_format='{l_bar}{bar:30}{r_bar}')

for epoch in ebar:
    model.train(); tr_losses = []
    bbar = tqdm(train_loader, desc=f'  E{epoch:03d}[Trn]', leave=False, bar_format='{l_bar}{bar:20}{r_bar}')
    for pm25, aux, tgt, wu, wv, ep_mask in bbar:
        pm25=pm25.to(cfg.DEVICE); aux=aux.to(cfg.DEVICE)
        tgt=tgt.to(cfg.DEVICE);   wu=wu.to(cfg.DEVICE)
        wv=wv.to(cfg.DEVICE);     ep_mask=ep_mask.to(cfg.DEVICE)
        optimizer.zero_grad()
        pred = model(pm25, aux)
        loss = total_loss(pred, tgt, wu, wv, ep_mask)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.GRAD_CLIP)
        optimizer.step()
        tr_losses.append(loss.item()); bbar.set_postfix(loss=f'{loss.item():.4f}')
    bbar.close(); scheduler.step()

    model.eval(); val_scores = []
    with torch.no_grad():
        for pm25, aux, tgt, wu, wv, ep_mask in tqdm(val_loader, desc=f'  E{epoch:03d}[Val]',
                                                     leave=False, bar_format='{l_bar}{bar:20}{r_bar}'):
            pred = model(pm25.to(cfg.DEVICE), aux.to(cfg.DEVICE))
            val_scores.append(val_metric_smape(pred, tgt))

    mean_val = float(np.mean(val_scores)); mean_tr = float(np.mean(tr_losses))
    if mean_val < best_val:
        best_val = mean_val
        best_state = {k: v.cpu().clone() for k,v in model.state_dict().items()}
        es_counter = 0
        tqdm.write(f'  🏆 E{epoch:03d} | tr={mean_tr:.4f} val_smape={best_val:.4f} ← BEST')
    else:
        es_counter += 1
        tqdm.write(f'  ⏳ E{epoch:03d} | tr={mean_tr:.4f} val_smape={mean_val:.4f} [{es_counter}/{cfg.ES_PATIENCE}]')

    ebar.set_postfix(val=f'{mean_val:.4f}', best=f'{best_val:.4f}',
                     pat=f'{es_counter}/{cfg.ES_PATIENCE}')

    if epoch % 5 == 0:
        torch.save({'epoch':epoch,'model':model.state_dict(),'optimizer':optimizer.state_dict(),
                    'scheduler':scheduler.state_dict(),'best_val':best_val,'best_state':best_state,
                    'es_counter':es_counter}, cfg.CKPT_PATH)
        tqdm.write(f'  💾 Checkpoint epoch {epoch}')

    if es_counter >= cfg.ES_PATIENCE:
        tqdm.write(f'\n🛑 Early stop epoch {epoch} | Best SMAPE: {best_val:.4f}'); break

ebar.close()
torch.save(best_state, cfg.MODEL_PATH)
print(f"\n✅ Best model saved | val SMAPE: {best_val:.4f}")


Fresh start | seed=123 | 100 epochs | ES=20


Training:   0%|                              | 0/100 [00:00<?, ?ep/s]

  E001[Trn]:   0%|                    | 0/601 [00:00<?, ?it/s]

  E001[Val]:   0%|                    | 0/54 [00:00<?, ?it/s]

  🏆 E001 | tr=0.0671 val_smape=0.2643 ← BEST


...

  E090[Val]:   0%|                    | 0/54 [00:00<?, ?it/s]

  ⏳ E090 | tr=0.0178 val_smape=0.1706 [19/20]
  💾 Checkpoint epoch 90


  E091[Trn]:   0%|                    | 0/601 [00:00<?, ?it/s]

  E091[Val]:   0%|                    | 0/54 [00:00<?, ?it/s]

  ⏳ E091 | tr=0.0177 val_smape=0.1700 [20/20]

🛑 Early stop epoch 91 | Best SMAPE: 0.1691

✅ Best model saved | val SMAPE: 0.1691


In [9]:
del month_arrays, train_ds, val_ds, train_loader, val_loader
del train_indices, val_indices
torch.cuda.empty_cache(); gc.collect()
import psutil
print(f"✅ RAM cleared | Free: {psutil.virtual_memory().available/1024**3:.1f} GB")


✅ RAM cleared | Free: 23.9 GB


In [10]:
gc.collect(); torch.cuda.empty_cache()

N, BATCH = 218, 8   # Phase 2: 218 test samples (NOT 996)

def load_weights(path):
    raw = torch.load(path, map_location=cfg.DEVICE)
    return OrderedDict((k.replace('module.',''),v) for k,v in raw.items() if k != 'n_averaged')

model = PTM_FNO_Phase2().to(cfg.DEVICE)
model.load_state_dict(load_weights(cfg.MODEL_PATH))
model.eval()
print("✅ Model loaded")

# ── Load test inputs ──────────────────────────────────────────
# Phase 2: ALL features have T=10 only (no future met)
pm25_raw  = np.load(os.path.join(cfg.TEST_PATH, 'cpm25.npy')).astype(np.float32)
pm25_norm = norm_pm25(pm25_raw)
del pm25_raw; gc.collect()
print(f"✅ Test PM2.5: {pm25_norm.shape}")  # expect (218, 10, 140, 124)

aux_mmap = {feat: np.load(os.path.join(cfg.TEST_PATH, f'{feat}.npy'), mmap_mode='r')
            for feat in cfg.ALL_AUX}
print(f"✅ Aux features: {len(aux_mmap)} × shape {aux_mmap['q2'].shape}")  # expect (218, 10, H, W)

all_preds = []
for i in range(0, N, BATCH):
    aux_list = []
    for feat in cfg.ALL_AUX:
        arr = aux_mmap[feat][i:i+BATCH].astype(np.float32)  # (B, 10, H, W)
        arr = (arr - FEAT_MINS[feat]) / (FEAT_MAXS[feat] - FEAT_MINS[feat] + 1e-8)
        aux_list.append(arr)
    # Stack to (B, 10, 15, H, W)
    aux_b  = np.stack(aux_list, axis=2).astype(np.float32)
    pm25_b = pm25_norm[i:i+BATCH]   # (B, 10, H, W)
    pm25_t = torch.from_numpy(pm25_b).float().to(cfg.DEVICE)
    aux_t  = torch.from_numpy(aux_b).float().to(cfg.DEVICE)
    with torch.no_grad():
        pred_b = model(pm25_t, aux_t)
    all_preds.append(pred_b.cpu().float().numpy())
    del pm25_t, aux_t; torch.cuda.empty_cache()
    if i % 50 == 0: print(f"  {i}/{N}...")

preds_norm = np.concatenate(all_preds, axis=0)   # (218, 16, 140, 124)
preds      = np.clip(denorm_pm25_np(preds_norm), 0, 500)
preds      = preds.transpose(0, 2, 3, 1)          # (218, 140, 124, 16)

assert preds.shape == (218, 140, 124, 16), f"Shape error: {preds.shape}"
assert np.isfinite(preds).all(), "NaN/Inf!"
assert preds.min() >= 0

np.save(cfg.OUTPUT_PATH, preds)
print(f"\n✅ Saved → {cfg.OUTPUT_PATH}")
print(f"   Shape={preds.shape}")
print(f"   Mean={preds.mean():.1f}  Std={preds.std():.1f}  Min={preds.min():.1f}  Max={preds.max():.1f} µg/m³")
print()
if 28 <= preds.mean() <= 45:
    print("  ✅ HEALTHY — SUBMIT THIS")
else:
    print(f"  ❌ Mean {preds.mean():.1f} looks off — check before submitting")


✅ Model loaded
✅ Test PM2.5: (218, 10, 140, 124)
✅ Aux features: 15 × shape (218, 10, 140, 124)
  0/218...
  200/218...

✅ Saved → /kaggle/working/preds.npy
   Shape=(218, 140, 124, 16)
   Mean=36.0  Std=49.1  Min=0.0  Max=500.0 µg/m³

  ✅ HEALTHY — SUBMIT THIS
